# Import required libraries and mount Google Drive

In [1]:
# notebooks/01_Data_Preparation.ipynb

import os
import zipfile
import json
import pandas as pd
from sklearn.model_selection import train_test_split
import sys
from google.colab import drive
from tqdm import tqdm

# Mount Google Drive to access persistent storage
drive.mount('/content/drive')

# Add src to path
PROJECT_ROOT = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor"
sys.path.append(PROJECT_ROOT)
import src.config as config

Mounted at /content/drive


# Extract the RAW dataset zip file to local storage

In [2]:
print("Extracting raw dataset zip file to local runtime storage...")
os.makedirs(config.DATA_ROOT_DIR_RAW, exist_ok=True)

with zipfile.ZipFile(config.ZIP_SOURCE_PATH_RAW, 'r') as zip_ref:
    unzip_targets = zip_ref.namelist()
    for file in tqdm(unzip_targets, desc="Extracting raw data"):
        zip_ref.extract(file, config.LOCAL_EXTRACT_DIR)

print(f"Extraction complete. Raw data directory: {config.DATA_ROOT_DIR_RAW}")

Extracting raw dataset zip file to local runtime storage...


Extracting raw data: 100%|██████████| 1847/1847 [05:36<00:00,  5.49it/s]

Extraction complete. Raw data directory: /content/MICCAI_BraTS2020_TrainingData


# Isotropic Bounding Box Crop, Resize to 128^3, and Save/Zip to Drive

In [3]:
import shutil
import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F

# Define paths to metadata
raw_name_mapping_csv = os.path.join(config.DATA_ROOT_DIR_RAW, "name_mapping.csv")
raw_survival_info_csv = os.path.join(config.DATA_ROOT_DIR_RAW, "survival_info.csv")

df_meta = pd.read_csv(raw_name_mapping_csv)
subject_ids = df_meta['BraTS_2020_subject_ID'].dropna().tolist()

# -------------------------------------------------------------------------
# PHASE 1: Find the largest brain dimension across the whole dataset
# -------------------------------------------------------------------------
print("Phase 1: Scanning dataset to determine global maximum brain bounding box (b)...")
max_spans = [0, 0, 0]

for subject_id in tqdm(subject_ids, desc="  Scanning for global b"):
    ref_path = os.path.join(config.DATA_ROOT_DIR_RAW, subject_id, f"{subject_id}_{config.MODALITIES[0]}.nii")
    if not os.path.exists(ref_path):
        continue

    img = nib.load(ref_path).get_fdata()
    non_zero_coords = np.argwhere(img > 0)
    if len(non_zero_coords) == 0:
        continue

    min_idx = non_zero_coords.min(axis=0)
    max_idx = non_zero_coords.max(axis=0)
    spans = max_idx - min_idx + 1

    for d in range(3):
        if spans[d] > max_spans[d]:
            max_spans[d] = spans[d]

b = int(max(max_spans))
margin = 1  # 1-voxel buffer to protect against odd/even rounding imbalances
B = b + 2 * margin
print(f"  Global b identified: {b}. Unified bounding box cube size B: {B}x{B}x{B}")

# -------------------------------------------------------------------------
# PHASE 2: Isotropic Bounding Box Crop, Resize, and Metadata Correction
# -------------------------------------------------------------------------
print("\nPhase 2: Executing Isotropic Bounding Box Crop and Resize Loop...")
for subject_id in tqdm(subject_ids, desc="  Preprocessing subjects"):
    subj_raw_dir = os.path.join(config.DATA_ROOT_DIR_RAW, subject_id)
    subj_out_dir = os.path.join(config.DATA_ROOT_DIR, subject_id)

    modality_paths = [os.path.join(subj_raw_dir, f"{subject_id}_{mod}.nii") for mod in config.MODALITIES]
    seg_path = os.path.join(subj_raw_dir, f"{subject_id}_seg.nii")

    if not (all(os.path.exists(p) for p in modality_paths) and os.path.exists(seg_path)):
        continue

    os.makedirs(subj_out_dir, exist_ok=True)

    # Load reference sequence header for physical space matrix mappings
    ref_nib = nib.load(modality_paths[0])
    ref_data = ref_nib.get_fdata()
    ref_affine = ref_nib.affine

    non_zero_coords = np.argwhere(ref_data > 0)
    if len(non_zero_coords) == 0:
        continue

    min_idx = non_zero_coords.min(axis=0)
    max_idx = non_zero_coords.max(axis=0)
    center_c = ((min_idx + max_idx) // 2).astype(int)

    half_B = B // 2
    start_coords = center_c - half_B
    end_coords = start_coords + B

    all_paths = modality_paths + [seg_path]
    for p in all_paths:
        file_nib = nib.load(p)
        file_data = file_nib.get_fdata()

        # Symmetrically handle matrix bounds overflows
        pad_before = np.maximum(0, -start_coords)
        pad_after = np.maximum(0, end_coords - np.array(file_data.shape))

        if np.any(pad_before > 0) or np.any(pad_after > 0):
            file_data = np.pad(
                file_data,
                ((pad_before[0], pad_after[0]),
                 (pad_before[1], pad_after[1]),
                 (pad_before[2], pad_after[2])),
                mode='constant', constant_values=0
            )
            adj_start = start_coords + pad_before
        else:
            adj_start = start_coords

        cropped_data = file_data[
            adj_start[0]:adj_start[0]+B,
            adj_start[1]:adj_start[1]+B,
            adj_start[2]:adj_start[2]+B
        ]

        tensor_data = torch.from_numpy(cropped_data).float().unsqueeze(0).unsqueeze(0)

        is_seg = "_seg.nii" in p
        mode = "nearest" if is_seg else "trilinear"

        resized_tensor = F.interpolate(
            tensor_data,
            size=(128, 128, 128),
            mode=mode,
            align_corners=None if is_seg else False
        )
        resized_data = resized_tensor.squeeze(0).squeeze(0).numpy()

        if is_seg:
            resized_data = resized_data.astype(np.int32)

        # -----------------------------------------------------------------
        # METADATA FIXES: Correct voxel spacing affine matrix & spatial origin
        # -----------------------------------------------------------------
        new_affine = np.copy(ref_affine)
        # Update spacing scales on the diagonal elements
        new_affine[:3, :3] = ref_affine[:3, :3] * (B / 128.0)
        # Shift origin matrix coordinates to preserve 3D visualization positioning
        new_affine[:3, 3] = ref_affine[:3, :3] @ adj_start + ref_affine[:3, 3]

        out_file_path = os.path.join(subj_out_dir, os.path.basename(p))
        new_nib = nib.Nifti1Image(resized_data, new_affine)

        # Explicitly apply corrected pixdim zooms directly inside the NIfTI headers
        new_zooms = [zoom * B / 128.0 for zoom in file_nib.header.get_zooms()[:3]]
        new_nib.header.set_zooms(new_zooms)

        nib.save(new_nib, out_file_path)

# -------------------------------------------------------------------------
# PHASE 3: Package preprocessed records and transfer back to Drive storage
# -------------------------------------------------------------------------
print("\nPhase 3: Copying Metadata CSV Records to Preprocessed Target Directory...")
shutil.copy(raw_name_mapping_csv, config.NAME_MAPPING_CSV)
if os.path.exists(raw_survival_info_csv):
    shutil.copy(raw_survival_info_csv, config.SURVIVAL_INFO_CSV)

print("\nPhase 4: Compressing and Saving Preprocessed Dataset Zip back to Drive...")
drive_zip_dir = os.path.dirname(config.ZIP_SOURCE_PATH)
os.makedirs(drive_zip_dir, exist_ok=True)
zip_base_name = os.path.splitext(config.ZIP_SOURCE_PATH)[0]
shutil.make_archive(zip_base_name, 'zip', config.DATA_ROOT_DIR)

print(f"  Complete preprocessing cycle finished! Archive saved to drive at: {config.ZIP_SOURCE_PATH}")

Phase 1: Scanning dataset to determine global maximum brain bounding box (b)...


  Scanning for global b: 100%|██████████| 369/369 [01:26<00:00,  4.27it/s]


  Global b identified: 193. Unified bounding box cube size B: 195x195x195

Phase 2: Executing Isotropic Bounding Box Crop and Resize Loop...


  Preprocessing subjects: 100%|██████████| 369/369 [09:42<00:00,  1.58s/it]



Phase 3: Copying Metadata CSV Records to Preprocessed Target Directory...

Phase 4: Compressing and Saving Preprocessed Dataset Zip back to Drive...
  Complete preprocessing cycle finished! Archive saved to drive at: /content/drive/MyDrive/ML-Datasets/BraTS2020_TrainingData_128.zip


# Parse name_mapping.csv and prepare patient dictionaries

In [4]:
print("Parsing metadata and creating stratified splits...")
df = pd.read_csv(config.NAME_MAPPING_CSV)

# Map grades to binary integers (0: LGG, 1: HGG)
grade_map = {"LGG": 0, "HGG": 1}
df['grade_int'] = df['Grade'].map(grade_map)

patient_records = []
for idx, row in df.iterrows():
    subject_id = row['BraTS_2020_subject_ID']
    grade = row['grade_int']

    # Paths for each structural modality matching the specified sequence order
    modality_paths = [
        os.path.join(config.DATA_ROOT_DIR, subject_id, f"{subject_id}_{mod}.nii")
        for mod in config.MODALITIES
    ]

    # Path for the segmentation target mask
    seg_path = os.path.join(config.DATA_ROOT_DIR, subject_id, f"{subject_id}_seg.nii")

    # Check all 5 files exist before adding to records
    files_exist = all(os.path.exists(p) for p in modality_paths) and os.path.exists(seg_path)
    if not files_exist:
        print(f"Warning: Missing files for subject {subject_id}. Skipping.")
        continue

    patient_records.append({
        "image": modality_paths,
        "label": seg_path,
        "grade": int(grade)
    })

Parsing metadata and creating stratified splits...


# Execute stratified data splitting to preserve HGG/LGG distribution ratio

In [5]:
grades = [record['grade'] for record in patient_records]

# Separate into 80% train and 20% val/test combined
train_records, val_test_records = train_test_split(
    patient_records,
    test_size=(config.VAL_RATIO + config.TEST_RATIO),
    stratify=grades,
    random_state=config.RANDOM_SEED
)

# Split the remaining 20% equally into vali (10%) and test (10%)
val_test_grades = [record['grade'] for record in val_test_records]
val_records, test_records = train_test_split(
    val_test_records,
    test_size=0.5,
    stratify=val_test_grades,
    random_state=config.RANDOM_SEED
)

print(f"Split distribution summary:")
print(f"Train samples: {len(train_records)}")
print(f"Validation samples: {len(val_records)}")
print(f"Test samples: {len(test_records)}")

Split distribution summary:
Train samples: 294
Validation samples: 37
Test samples: 37


# Save dataset split configuration to a single JSON file

In [6]:
split_data = {
    "train": train_records,
    "val": val_records,
    "test": test_records
}

os.makedirs(os.path.dirname(config.DATA_SPLIT_JSON), exist_ok=True)

with open(config.DATA_SPLIT_JSON, 'w') as f:
    json.dump(split_data, f, indent=4)

print(f"Successfully saved data splits configuration to {config.DATA_SPLIT_JSON}")

Successfully saved data splits configuration to /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/data_splits.json
